In [3]:
import sys
import os
import glob
import librosa
import numpy as np

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf

In [4]:
def file_to_vector_array(file_path, n_mels=128, frames=5, n_fft=1024, hop_length=512, power=2.0):
    dims = n_mels * frames

    y, sr = librosa.load(file_path, sr=None, mono=True)
    mel_spectrogram = librosa.feature.melspectrogram(y=y,
                                                     sr=sr,
                                                     n_fft=n_fft,
                                                     hop_length=hop_length,
                                                     n_mels=n_mels,
                                                     power=power)

    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max).astype(np.float32)

    vector_array_size = len(log_mel_spectrogram[0, :]) - frames + 1

    if vector_array_size < 1:
        return np.empty((0, dims))

    vector_array = np.zeros((vector_array_size, dims))
    for t in range(frames):
        vector_array[:, n_mels * t: n_mels * (t + 1)] = log_mel_spectrogram[:, t: t + vector_array_size].T

    return vector_array

def list_to_vector_array(file_list, n_mels=128, frames=5, n_fft=1024, hop_length=512, power=2.0):
    all_vectors = []
    for file in file_list:
        vectors = file_to_vector_array(file, n_mels=n_mels, frames=frames, n_fft=n_fft, hop_length=hop_length, power=power)
        all_vectors.append(vectors)

    return np.vstack(all_vectors)

In [5]:
import keras.models
from keras.models import Model
from keras.layers import Input, Dense, BatchNormalization, Activation

def get_model(inputDim):
    inputLayer = Input(shape=(inputDim,))

    h = Dense(512)(inputLayer)
    h = BatchNormalization()(h)
    h = Activation('relu')(h)

    h = Dense(512)(h)
    h = BatchNormalization()(h)
    h = Activation('relu')(h)

    h = Dense(512)(h)
    h = BatchNormalization()(h)
    h = Activation('relu')(h)

    h = Dense(512)(h)
    h = BatchNormalization()(h)
    h = Activation('relu')(h)

    h = Dense(8)(h)
    h = BatchNormalization()(h)
    h = Activation('relu')(h)

    h = Dense(512)(h)
    h = BatchNormalization()(h)
    h = Activation('relu')(h)

    h = Dense(512)(h)
    h = BatchNormalization()(h)
    h = Activation('relu')(h)

    h = Dense(512)(h)
    h = BatchNormalization()(h)
    h = Activation('relu')(h)

    h = Dense(512)(h)
    h = BatchNormalization()(h)
    h = Activation('relu')(h)

    h = Dense(inputDim)(h)

    return Model(inputs=inputLayer, outputs=h)

In [6]:
def list_wavs_by_machine_id(train_dir, machine_id):
    training_list_path = os.path.relpath("../dataset/{dir}/*_id_{id}_*.wav".format(dir=train_dir, id=machine_id))
    files = sorted(glob.glob(training_list_path))
    return files

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, RocCurveDisplay
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt

machine_config = {
    'fan': ['00', '02', '04', '06'],
    'valve': ['00', '02', '04', '06'],
    'pump': ['00', '02', '04', '06'],
    'slider': ['00', '02', '04', '06'],
    'ToyCar': ['01', '02', '03', '04'],
    'ToyConveyor': ['01', '02', '03']
}

baseline_results = []

for machine_type, machine_ids in machine_config.items():
    for m_id in machine_ids:
        print(f'--- (Baseline) Processing {machine_type} with ID: {m_id} ---')
        
        train_files = list_wavs_by_machine_id(f'{machine_type}/train', m_id)

        train_files, val_files = train_test_split(train_files, test_size=0.1, random_state=42)

        train_data = list_to_vector_array(train_files)
        val_data = list_to_vector_array(val_files)
        
        scaler = StandardScaler()
        train_data = scaler.fit_transform(train_data)

        val_data = scaler.transform(val_data)
        
        model = get_model(128 * 5)
        model.compile(optimizer='adam', loss='mse')

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=10, verbose=1, restore_best_weights=True),
            ModelCheckpoint(filepath=f'../models/baseline/baseline_model_{machine_type}_{m_id}.keras', monitor='val_loss', mode='min', save_best_only=True)
        ]
        
        model.fit(train_data, train_data,
            epochs=100,
            batch_size=512,
            validation_data=(val_data, val_data),
            callbacks=callbacks
        )
        
        test_files = list_wavs_by_machine_id(f'{machine_type}/test', m_id)
        y_true = [1 if 'anomaly' in os.path.basename(f) else 0 for f in test_files]
        y_pred_scores = []
        
        for file_path in test_files:
            vectors = file_to_vector_array(file_path)
            vectors = scaler.transform(vectors)
            reconstructed = model.predict(vectors, verbose=0)
            mse = np.mean(np.square(vectors - reconstructed), axis=1)
            y_pred_scores.append(np.mean(mse))
        
        fpr, tpr, _ = roc_curve(y_true, y_pred_scores)
        roc_auc = auc(fpr, tpr)

        baseline_results.append({
            'Machine_Type': machine_type,
            'Machine_ID': m_id,
            'AUC_Score': roc_auc
        })

        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC - {machine_type} with ID {m_id}')
        plt.legend(loc="lower right")
        plt.grid(alpha=0.3)
        
        plt.savefig(f'../results/baseline_autoencoder/{machine_type}_{m_id}_baseline_roc.png')
        plt.close()

--- (Baseline) Processing fan with ID: 00 ---
Epoch 1/100
495/495 ━━━━━━━━━━━━━━━━━━━━ 14s 14ms/step - loss: 0.3798 - val_loss: 0.3787
Epoch 2/100
495/495 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.3347 - val_loss: 0.3412
Epoch 3/100
495/495 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.3253 - val_loss: 0.3371
Epoch 4/100
495/495 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.3207 - val_loss: 0.3320
Epoch 5/100
495/495 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.3175 - val_loss: 0.3304
Epoch 6/100
495/495 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.3151 - val_loss: 0.3314
Epoch 7/100
495/495 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.3130 - val_loss: 0.3282
Epoch 8/100
495/495 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.3110 - val_loss: 0.3270

In [8]:
results_df = pd.DataFrame(baseline_results)
results_df

,Machine_Type,Machine_ID,AUC_Score
0,fan,00,0.573145
1,fan,02,0.783844
2,fan,04,0.625259
3,fan,06,0.863019
4,valve,00,0.707143
5,valve,02,0.710417
6,valve,04,0.762417
7,valve,06,0.620000
8,pump,00,0.712517
9,pump,02,0.635495
